# Chapter 7 — Near-Earth Propagation Models — equations

Standalone, runnable subset of the master `../RF_Equations.ipynb`, scoped to this chapter.
Run top-to-bottom: **Setup**, then this chapter's sections. All functions are verified against the book's worked examples.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


## 14. Near-Earth Models: Foliage, Terrain, Urban — Ch 7 *(outdoor city track)*

The empirical baseline + validation reference for the **outdoor OSM-voxelized sim**. All are
*median* path-loss fits (not physics). **Hata / COST-231 are the workhorses** for mobile urban.

- Foliage: Weissberger `1.33 F^0.284 d^0.588` (14–400 m) / `0.45 F^0.284 d` (<14 m), F in GHz
  (eq 7.1); early ITU `0.2 F^0.3 d^0.6`, F in MHz (eq 7.2). → `weissberger_foliage_db()`, `itu_foliage_db()`.
- Terrain: **Egli** 4th-power law `(hb hm/d²)²(40/f)²` (eq 7.9) → `egli_pl_db()`; **ITU terrain
  diffraction** `Ad = −20 h/F1 + 10` with `F1 = 17.3√(d1 d2/(f d))` (the Ch 8 Fresnel radius in
  km/GHz) → `itu_terrain_diffraction_db()`.
- **Urban macro-models:** **Hata** (150–1500 MHz, eq 7.14) urban/suburban/open → `hata_pl_db()`;
  **COST-231** (1500–2000 MHz PCS, eq 7.19) → `cost231_pl_db()`; **Lee** (fittable power law,
  eq 7.20) → `lee_pl_db()`. Okumura = the graphical parent of Hata; Young = NYC power law.


In [ ]:
def weissberger_foliage_db(d_f_m, f_ghz):
    if d_f_m <= 14: return 0.45*f_ghz**0.284*d_f_m               # 0 < d <= 14 m
    return 1.33*f_ghz**0.284*d_f_m**0.588                        # 14 < d <= 400 m

def itu_foliage_db(d_f_m, f_mhz):
    return 0.2*f_mhz**0.3*d_f_m**0.6                             # eq 7.2 (F in MHz)

def egli_pl_db(d_m, f_mhz, hb_m, hm_m, Gb=1.0, Gm=1.0):
    gain = Gb*Gm*(hb_m*hm_m/d_m**2)**2*(40.0/f_mhz)**2           # Pr/Pt (eq 7.9)
    return -10*np.log10(gain)

def fresnel_radius_terrain_m(d1_km, d2_km, d_km, f_ghz):
    return 17.3*np.sqrt(d1_km*d2_km/(f_ghz*d_km))               # eq 7.11 (= Ch 8 Fresnel radius)

def itu_terrain_diffraction_db(h_m, F1_m):
    return -20*(h_m/F1_m) + 10                                   # eq 7.10

# Example 7.1: 12 m of trees, 1 GHz
print(f"Ex 7.1: Weissberger={weissberger_foliage_db(12, 1.0):.2f} dB (book 5.4), "
      f"ITU={itu_foliage_db(12, 1000):.2f} dB (book 7.06)")
# Example 7.3: 3 km, 100 MHz, blockage mid-path, h=0.75 m
F1 = fresnel_radius_terrain_m(1.5, 1.5, 3.0, 0.1)
print(f"Ex 7.3: F1={F1:.1f} m (book 47.4), Ad={itu_terrain_diffraction_db(0.75, F1):.1f} dB (book 9.7)")
# Egli Example 7.2: 100 MHz, hb=20, hm=3, d=1 km
print(f"Ex 7.2: Egli={egli_pl_db(1000, 100, 20, 3):.1f} dB  "
      f"(book prints 112.4, but that used hb*hm=6 instead of 60; correct value is 92.4)")


In [ ]:
def hata_a_hr(hr_m, fc_mhz, large_city=False):
    if large_city:
        if fc_mhz <= 200: return 8.29*(np.log10(1.54*hr_m))**2 - 1.1
        return 3.2*(np.log10(11.75*hr_m))**2 - 4.97             # fc >= 400 MHz
    return (1.1*np.log10(fc_mhz) - 0.7)*hr_m - (1.56*np.log10(fc_mhz) - 0.8)

def hata_pl_db(d_km, fc_mhz, ht_m, hr_m, area="urban", large_city=False):
    a = hata_a_hr(hr_m, fc_mhz, large_city)
    urban = (69.55 + 26.16*np.log10(fc_mhz) - 13.82*np.log10(ht_m) - a
             + (44.9 - 6.55*np.log10(ht_m))*np.log10(d_km))       # eq 7.14
    if area == "suburban": return urban - 2*(np.log10(fc_mhz/28.0))**2 - 5.4      # eq 7.17
    if area == "open":     return urban - 4.78*(np.log10(fc_mhz))**2 + 18.33*np.log10(fc_mhz) - 40.94  # eq 7.18
    return urban

def cost231_pl_db(d_km, fc_mhz, ht_m, hr_m, metro=False, large_city=False):
    a = hata_a_hr(hr_m, fc_mhz, large_city)
    return (46.3 + 33.9*np.log10(fc_mhz) - 13.82*np.log10(ht_m) - a
            + (44.9 - 6.55*np.log10(ht_m))*np.log10(d_km) + (3.0 if metro else 0.0))  # eq 7.19

LEE_PARAMS = {   # environment: (L0_dB at 1 km, gamma dB/decade)  -- Table 7.2
    "free_space": (85, 20), "open_rural": (89, 43.5), "suburban": (101.7, 38.5),
    "philadelphia": (110, 36.8), "newark": (104, 43.1), "tokyo": (124.0, 30.5),
}
def lee_pl_db(d_km, L0_db, gamma, F0_db=0.0):
    return L0_db + gamma*np.log10(d_km) - F0_db                   # eq 7.20 (F0 already in dB)

# Example 7.5: ht=68, fc=870 MHz, hr=3, d=3.7 km, large city
print(f"Ex 7.5: Hata urban(large city) = {hata_pl_db(3.7, 870, 68, 3, large_city=True):.1f} dB (book 137.1)")
print(f"        a(hr) = {hata_a_hr(3, 870, True):.2f} (book 2.69)")
# COST-231 sanity (same geometry, PCS band 1800 MHz)
print(f"COST-231 @1800 MHz (metro) = {cost231_pl_db(3.7, 1800, 68, 3, metro=True, large_city=True):.1f} dB")
# Example 7.6 (Lee, suburban): L0=101.7, gamma=38.5, F0=-5 dB -> 106.7 + 38.5 log d
print(f"Ex 7.6: Lee suburban = {lee_pl_db(1.0, 101.7, 38.5, -5.0):.1f} + 38.5*log(d)  (book 106.7 + 38.5 log d)")
